In [1]:
# Direct download from original source (Institut für Neuroinformatik)
!wget https://sid.erda.dk/public/archives/daaeac0d7ce1152aea9b61d9f1e19370/GTSRB_Final_Training_Images.zip
#!wget https://sid.erda.dk/public/archives/daaeac0d7ce1152aea9b61d9f1e19370/GTSRB_Final_Test_Images.zip

# Unzip both files
!unzip GTSRB_Final_Training_Images.zip
#!unzip GTSRB_Final_Test_Images.zip

# Cleanup (optional)
!rm *.zip

print("Dataset downloaded and extracted!")

Streaming output truncated to the last 5000 lines.
  inflating: GTSRB/Final_Training/Images/00035/00000_00020.ppm  
  inflating: GTSRB/Final_Training/Images/00035/00000_00021.ppm  
  inflating: GTSRB/Final_Training/Images/00035/00000_00022.ppm  
  inflating: GTSRB/Final_Training/Images/00035/00000_00023.ppm  
  inflating: GTSRB/Final_Training/Images/00035/00000_00024.ppm  
  inflating: GTSRB/Final_Training/Images/00035/00000_00025.ppm  
  inflating: GTSRB/Final_Training/Images/00035/00000_00026.ppm  
  inflating: GTSRB/Final_Training/Images/00035/00000_00027.ppm  
  inflating: GTSRB/Final_Training/Images/00035/00000_00028.ppm  
  inflating: GTSRB/Final_Training/Images/00035/00000_00029.ppm  
  inflating: GTSRB/Final_Training/Images/00035/00001_00000.ppm  
  inflating: GTSRB/Final_Training/Images/00035/00001_00001.ppm  
  inflating: GTSRB/Final_Training/Images/00035/00001_00002.ppm  
  inflating: GTSRB/Final_Training/Images/00035/00001_00003.ppm  
  inflating: GTSRB/Final_Training/Image

In [2]:
import os
from PIL import Image
import shutil

# Define paths
original_dir = '/content/GTSRB/Final_Training/Images'
processed_dir = '/content/GTSRB_Train'

# Create output directory if it doesn't exist
os.makedirs(processed_dir, exist_ok=True)

# Walk through all class folders
for class_id in os.listdir(original_dir):
    class_path = os.path.join(original_dir, class_id)
    if not os.path.isdir(class_path):
        continue  # skip non-directories

    # Create corresponding class folder in processed_dir
    output_class_path = os.path.join(processed_dir, class_id)
    os.makedirs(output_class_path, exist_ok=True)

    # Process each PPM file
    for filename in os.listdir(class_path):
        if filename.endswith('.ppm'):
            img_path = os.path.join(class_path, filename)
            img = Image.open(img_path)
            # Save as .png
            new_filename = os.path.splitext(filename)[0] + '.png'
            img.save(os.path.join(output_class_path, new_filename))

print("All .ppm files converted and saved as .png in:", processed_dir)


All .ppm files converted and saved as .png in: /content/GTSRB_Train


In [3]:
# Step 1: Mount your Drive
from google.colab import drive
drive.mount('/content/drive')

# Step 2: Navigate to where you saved the file
model_path = '/content/drive/MyDrive/Effnet Adverserial_Training/EfficientNetB1.keras'

# Step 3: Load the model
from tensorflow.keras.models import load_model
model = load_model(model_path)

print("Model loaded successfully!")


Mounted at /content/drive
Model loaded successfully!


# Fine-Tuning Model on Gaussian Distribuition

In [4]:
import tensorflow as tf
from tensorflow.keras.models import load_model

# 1) Load your pretrained EfficientNetB1
model = load_model(model_path)

# 2) Build train & validation datasets with an 80/20 split
base_dir   = '/content/GTSRB_Train'
batch_size = 32
img_size   = (240, 240)
seed       = 123

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    base_dir,
    labels='inferred',
    label_mode='categorical',
    batch_size=batch_size,
    image_size=img_size,
    shuffle=True,
    validation_split=0.2,
    subset='training',
    seed=seed
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    base_dir,
    labels='inferred',
    label_mode='categorical',
    batch_size=batch_size,
    image_size=img_size,
    shuffle=False,
    validation_split=0.2,
    subset='validation',
    seed=seed
)

# 3) Define a fixed-amplitude Gaussian noise fusion and random batch-wise toggle
NOISE_AMP = 0.2

def maybe_noisy(images, labels):
    """
    With probability 0.5, add Gaussian noise of amplitude NOISE_AMP to the batch;
    otherwise leave the batch clean. Keeps total number of samples unchanged.
    """
    images = tf.cast(images, tf.float32)
    do_noise = tf.less(tf.random.uniform([], 0, 1), 0.5)

    def add_noise():
        noise = tf.random.normal(shape=tf.shape(images), mean=0.0, stddev=255.0)
        fused = images * (1.0 - NOISE_AMP) + noise * NOISE_AMP
        fused = tf.clip_by_value(fused, 0.0, 255.0)
        return tf.cast(fused, images.dtype), labels

    clean = (tf.cast(images, images.dtype), labels)
    return tf.cond(do_noise, add_noise, lambda: clean)

# 4) Apply the random clean/noisy toggle to the single train_ds
alt_train_ds = (
    train_ds
    .map(maybe_noisy, num_parallel_calls=tf.data.AUTOTUNE)
    .prefetch(tf.data.AUTOTUNE)
)

# 5) Prefetch validation (no noise)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)

# 6) Compile & fine-tune
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    alt_train_ds,
    epochs=10,           # adjust as needed
    validation_data=val_ds
)


Found 39209 files belonging to 43 classes.
Using 31368 files for training.
Found 39209 files belonging to 43 classes.
Using 7841 files for validation.
Epoch 1/10
981/981 ━━━━━━━━━━━━━━━━━━━━ 374s 270ms/step - accuracy: 0.9560 - loss: 0.1779 - val_accuracy: 0.9987 - val_loss: 0.0042
Epoch 2/10
981/981 ━━━━━━━━━━━━━━━━━━━━ 181s 184ms/step - accuracy: 0.9813 - loss: 0.0621 - val_accuracy: 0.9997 - val_loss: 0.0012
Epoch 3/10
981/981 ━━━━━━━━━━━━━━━━━━━━ 183s 186ms/step - accuracy: 0.9877 - loss: 0.0390 - val_accuracy: 0.9999 - val_loss: 0.0010
Epoch 4/10
981/981 ━━━━━━━━━━━━━━━━━━━━ 197s 181ms/step - accuracy: 0.9912 - loss: 0.0266 - val_accuracy: 0.9997 - val_loss: 7.9844e-04
Epoch 5/10
981/981 ━━━━━━━━━━━━━━━━━━━━ 182s 185ms/step - accuracy: 0.9921 - loss: 0.0230 - val_accuracy: 0.9997 - val_loss: 6.6441e-04
Epoch 6/10
981/981 ━━━━━━━━━━━━━━━━━━━━ 200s 184ms/step - accuracy: 0.9937 - loss: 0.0191 - val_accuracy: 0.9996 - val_loss: 8.3530e-04
Epoch 7/10
981/981 ━━━━━━━━━━━━━━━━━━━━ 202s 

In [5]:
# 8) Save the fine-tuned model
save_path = "/content/EfficientNetB1_NFM_finetuned_V2.keras"
model.save(save_path)
print(f"Model fine-tuned and saved at: {save_path}")

Model fine-tuned and saved at: /content/EfficientNetB1_NFM_finetuned_V2.keras


In [6]:
import gdown
import zipfile

url = "https://drive.google.com/uc?id=1tt6pE0OpFrykOOhRRSVYfKKSQRp8MbsB"
output_path = "Fgsm_test.zip"
gdown.download(url, output_path, quiet=False)

# Unzip
with zipfile.ZipFile(output_path, 'r') as zip_ref:
    zip_ref.extractall("Fgsm_test")

Downloading...
From (original): https://drive.google.com/uc?id=1tt6pE0OpFrykOOhRRSVYfKKSQRp8MbsB
From (redirected): https://drive.google.com/uc?id=1tt6pE0OpFrykOOhRRSVYfKKSQRp8MbsB&confirm=t&uuid=10ebd1c8-ef7e-4189-a0b2-42f3033a8274
To: /content/Fgsm_test.zip
100%|██████████| 175M/175M [00:03<00:00, 58.1MB/s]


In [7]:
# Install gdown if not already installed
!pip install gdown

# Download the file from Google Drive
file_id = "1avRnoqSRSIpDiHL9_JrZe-UlE3pZwmMx"
!gdown --id {file_id}

# Check the downloaded filename (replace 'filename.zip' with the actual name)
!ls

# Unzip the file into a folder named Pgd_test
!unzip -q adversarial_images_PGD.zip -d Pgd_test  # Replace 'filename.zip' with the actual name

/usr/local/lib/python3.11/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1avRnoqSRSIpDiHL9_JrZe-UlE3pZwmMx
From (redirected): https://drive.google.com/uc?id=1avRnoqSRSIpDiHL9_JrZe-UlE3pZwmMx&confirm=t&uuid=5f3183e1-d8b1-41fe-a398-5a11aba71321
To: /content/adversarial_images_PGD.zip
100% 206M/206M [00:06<00:00, 34.1MB/s]
adversarial_images_PGD.zip	       Fgsm_test      GTSRB_Train
drive				       Fgsm_test.zip  sample_data
EfficientNetB1_NFM_finetuned_V2.keras  GTSRB


In [8]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Load the CSV file
csv_path = '/content/Fgsm_test/adversarial_images/adversarial_labels.csv'
df = pd.read_csv(csv_path)

# Define the noise fusion function
def fuse_with_noise_batch(batch, alpha=0.2):
    batch = tf.cast(batch, tf.float32)
    noise = tf.random.normal(tf.shape(batch), mean=0.0, stddev=255.0)
    fused = batch * (1.0 - alpha) + noise * alpha
    fused = tf.clip_by_value(fused, 0.0, 255.0)
    return fused  # Stay float32

# List of epsilon values to evaluate
epsilons = [0.007, 0.01, 0.03, 0.1]

for eps in epsilons:
    print(f"\n{'='*50}\nEvaluating Adversarial Images for Epsilon = {eps}\n{'='*50}")

    # Filter CSV for the current epsilon
    eps_df = df[df['epsilon'] == eps].copy()

    # Remove the 'eps_X.XXX/' prefix from filenames (if present)
    eps_df['filename'] = eps_df['filename'].str.replace(f'eps_{eps}/', '', regex=False)

    # Create ImageDataGenerator (no normalization)
    test_datagen = ImageDataGenerator()

    # Prepare test generator
    test_generator = test_datagen.flow_from_dataframe(
        dataframe=eps_df,
        directory=f'/content/Fgsm_test/adversarial_images/eps_{eps}',  # Folder for current epsilon
        x_col='filename',
        y_col='label',
        target_size=(240, 240),
        batch_size=32,
        class_mode='raw',  # For integer labels
        shuffle=False
    )

    # Skip if no images found
    if test_generator.samples == 0:
        print(f" No images found for epsilon={eps}. Check paths or filenames.")
        continue

    # Predict batch-by-batch with noise fusion
    y_true = []
    y_pred = []

    for batch_x, batch_y in test_generator:
        # Apply noise fusion
        fused_batch_x = fuse_with_noise_batch(batch_x)

        # Predict
        preds = model.predict(fused_batch_x, verbose=0)
        batch_preds = tf.argmax(preds, axis=1).numpy()

        y_true.append(batch_y)
        y_pred.append(batch_preds)

        # Exit when all samples processed
        if len(np.concatenate(y_true)) >= test_generator.samples:
            break

    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)

    # Calculate metrics
    acc = accuracy_score(y_true, y_pred)
    print(f"\nPost-Adversarial Accuracy (with Noise Fusion, ε={eps}): {acc:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))



Evaluating Adversarial Images for Epsilon = 0.007
Found 516 validated image filenames.

Post-Adversarial Accuracy (with Noise Fusion, ε=0.007): 0.9516

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        12
           1       1.00      1.00      1.00        12
           2       1.00      1.00      1.00        12
           3       0.92      0.92      0.92        12
           4       1.00      1.00      1.00        12
           5       0.85      0.92      0.88        12
           6       1.00      0.83      0.91        12
           7       1.00      0.83      0.91        12
           8       0.79      0.92      0.85        12
           9       1.00      1.00      1.00        12
          10       1.00      1.00      1.00        12
          11       0.80      1.00      0.89        12
          12       0.92      0.92      0.92        12
          13       1.00      1.00      1.00        12
          14 

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



Post-Adversarial Accuracy (with Noise Fusion, ε=0.1): 0.4864

Classification Report:
              precision    recall  f1-score   support

           0       0.10      0.08      0.09        12
           1       0.29      0.92      0.44        12
           2       0.13      0.33      0.19        12
           3       1.00      0.08      0.15        12
           4       0.11      0.08      0.10        12
           5       0.00      0.00      0.00        12
           6       0.17      0.17      0.17        12
           7       0.50      0.08      0.14        12
           8       0.22      0.17      0.19        12
           9       0.24      0.58      0.34        12
          10       0.33      0.17      0.22        12
          11       0.44      0.67      0.53        12
          12       0.48      0.83      0.61        12
          13       0.86      1.00      0.92        12
          14       0.75      0.75      0.75        12
          15       1.00      0.50      0.67      

In [9]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Load the CSV file
csv_path = '/content/Pgd_test/adversarial_images_PGD/adversarial_labels.csv'
df = pd.read_csv(csv_path)

# Define the noise fusion function
def fuse_with_noise_batch(batch, alpha=0.2):
    batch = tf.cast(batch, tf.float32)
    noise = tf.random.normal(tf.shape(batch), mean=0.0, stddev=255.0)
    fused = batch * (1.0 - alpha) + noise * alpha
    fused = tf.clip_by_value(fused, 0.0, 255.0)
    return fused  # Stay float32

# List of epsilon values to evaluate
epsilons = [0.007, 0.01, 0.03, 0.1]

for eps in epsilons:
    print(f"\n{'='*50}\nEvaluating Adversarial Images for Epsilon = {eps}\n{'='*50}")

    # Filter CSV for the current epsilon
    eps_df = df[df['epsilon'] == eps].copy()

    # Remove the 'eps_X.XXX/' prefix from filenames (if present)
    eps_df['filename'] = eps_df['filename'].str.replace(f'eps_{eps}/', '', regex=False)

    # Create ImageDataGenerator (no normalization)
    test_datagen = ImageDataGenerator()

    # Prepare test generator
    test_generator = test_datagen.flow_from_dataframe(
        dataframe=eps_df,
        directory=f'/content/Pgd_test/adversarial_images_PGD/eps_{eps}',  # Folder for current epsilon
        x_col='filename',
        y_col='label',
        target_size=(240, 240),
        batch_size=32,
        class_mode='raw',  # For integer labels
        shuffle=False
    )

    # Skip if no images found
    if test_generator.samples == 0:
        print(f" No images found for epsilon={eps}. Check paths or filenames.")
        continue

    # Predict batch-by-batch with noise fusion
    y_true = []
    y_pred = []

    for batch_x, batch_y in test_generator:
        # Apply noise fusion
        fused_batch_x = fuse_with_noise_batch(batch_x)

        # Predict
        preds = model.predict(fused_batch_x, verbose=0)
        batch_preds = tf.argmax(preds, axis=1).numpy()

        y_true.append(batch_y)
        y_pred.append(batch_preds)

        # Exit when all samples processed
        if len(np.concatenate(y_true)) >= test_generator.samples:
            break

    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)

    # Calculate metrics
    acc = accuracy_score(y_true, y_pred)
    print(f"\nPost-Adversarial Accuracy (with Noise Fusion, ε={eps}): {acc:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))



Evaluating Adversarial Images for Epsilon = 0.007
Found 516 validated image filenames.

Post-Adversarial Accuracy (with Noise Fusion, ε=0.007): 0.9322

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.92      0.96        12
           1       1.00      1.00      1.00        12
           2       1.00      0.92      0.96        12
           3       0.92      0.92      0.92        12
           4       0.92      1.00      0.96        12
           5       0.65      0.92      0.76        12
           6       0.88      0.58      0.70        12
           7       1.00      1.00      1.00        12
           8       0.92      0.92      0.92        12
           9       1.00      0.92      0.96        12
          10       1.00      0.92      0.96        12
          11       0.86      1.00      0.92        12
          12       1.00      0.83      0.91        12
          13       1.00      1.00      1.00        12
          14 

# Fine-Tuning Model using Uniform distribuition Noise

In [10]:
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.callbacks import EarlyStopping

model_path_uniform = '/content/drive/MyDrive/Effnet Adverserial_Training/EfficientNetB1.keras'
model_uniform = load_model(model_path_uniform)

# 2) Build train & validation datasets with an 80/20 split
base_dir   = '/content/GTSRB_Train'
batch_size = 32
img_size   = (240, 240)
seed       = 123

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    base_dir,
    labels='inferred',
    label_mode='categorical',
    batch_size=batch_size,
    image_size=img_size,
    shuffle=True,
    validation_split=0.2,
    subset='training',
    seed=seed
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    base_dir,
    labels='inferred',
    label_mode='categorical',
    batch_size=batch_size,
    image_size=img_size,
    shuffle=False,
    validation_split=0.2,
    subset='validation',
    seed=seed
)

# 3) Define a fixed-amplitude Uniform noise fusion and random batch-wise toggle
NOISE_AMP = 0.2

def maybe_noisy(images, labels):
    images = tf.cast(images, tf.float32)
    do_noise = tf.less(tf.random.uniform([], 0, 1), 0.5)

    def add_noise():
        noise = tf.random.uniform(shape=tf.shape(images), minval=0, maxval=256, dtype=tf.float32)
        fused = images * (1.0 - NOISE_AMP) + noise * NOISE_AMP
        fused = tf.clip_by_value(fused, 0.0, 255.0)
        return tf.cast(fused, images.dtype), labels

    clean = (tf.cast(images, images.dtype), labels)
    return tf.cond(do_noise, add_noise, lambda: clean)

alt_train_ds = (
    train_ds
    .map(maybe_noisy, num_parallel_calls=tf.data.AUTOTUNE)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = val_ds.prefetch(tf.data.AUTOTUNE)

# 6) Compile & fine-tune
model_uniform.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# 7) Set up EarlyStopping on val_accuracy
early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=3,              # stop after 3 epochs with no improvement
    restore_best_weights=True
)

history = model_uniform.fit(
    alt_train_ds,
    epochs=10,               # you can raise this since ES will stop it early
    validation_data=val_ds,
    callbacks=[early_stop]   # <-- here
)


Found 39209 files belonging to 43 classes.
Using 31368 files for training.
Found 39209 files belonging to 43 classes.
Using 7841 files for validation.
Epoch 1/10
981/981 ━━━━━━━━━━━━━━━━━━━━ 341s 245ms/step - accuracy: 0.9910 - loss: 0.0305 - val_accuracy: 1.0000 - val_loss: 3.7543e-04
Epoch 2/10
981/981 ━━━━━━━━━━━━━━━━━━━━ 216s 186ms/step - accuracy: 0.9963 - loss: 0.0112 - val_accuracy: 0.9997 - val_loss: 3.8014e-04
Epoch 3/10
981/981 ━━━━━━━━━━━━━━━━━━━━ 202s 187ms/step - accuracy: 0.9977 - loss: 0.0063 - val_accuracy: 1.0000 - val_loss: 8.3785e-05
Epoch 4/10
981/981 ━━━━━━━━━━━━━━━━━━━━ 201s 186ms/step - accuracy: 0.9990 - loss: 0.0033 - val_accuracy: 1.0000 - val_loss: 1.4664e-04


In [11]:
# 8) Save the fine-tuned model
save_path = "/content/EfficientNetB1_NFM_finetuned_Uniform.keras"
model_uniform.save(save_path)
print(f"Model fine-tuned and saved at: {save_path}")

Model fine-tuned and saved at: /content/EfficientNetB1_NFM_finetuned_Uniform.keras


# Uniform distribuition on Fgsm

In [12]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Load the CSV file
csv_path = '/content/Fgsm_test/adversarial_images/adversarial_labels.csv'
df = pd.read_csv(csv_path)

# Define the noise fusion function
def fuse_with_noise_batch(batch, alpha=0.2):
    batch = tf.cast(batch, tf.float32)
    noise = tf.random.normal(tf.shape(batch), mean=0.0, stddev=255.0)
    fused = batch * (1.0 - alpha) + noise * alpha
    fused = tf.clip_by_value(fused, 0.0, 255.0)
    return fused  # Stay float32

# List of epsilon values to evaluate
epsilons = [0.007, 0.01, 0.03, 0.1]

for eps in epsilons:
    print(f"\n{'='*50}\nEvaluating Adversarial Images for Epsilon = {eps}\n{'='*50}")

    # Filter CSV for the current epsilon
    eps_df = df[df['epsilon'] == eps].copy()

    # Remove the 'eps_X.XXX/' prefix from filenames (if present)
    eps_df['filename'] = eps_df['filename'].str.replace(f'eps_{eps}/', '', regex=False)

    # Create ImageDataGenerator (no normalization)
    test_datagen = ImageDataGenerator()

    # Prepare test generator
    test_generator = test_datagen.flow_from_dataframe(
        dataframe=eps_df,
        directory=f'/content/Fgsm_test/adversarial_images/eps_{eps}',  # Folder for current epsilon
        x_col='filename',
        y_col='label',
        target_size=(240, 240),
        batch_size=32,
        class_mode='raw',  # For integer labels
        shuffle=False
    )

    # Skip if no images found
    if test_generator.samples == 0:
        print(f" No images found for epsilon={eps}. Check paths or filenames.")
        continue

    # Predict batch-by-batch with noise fusion
    y_true = []
    y_pred = []

    for batch_x, batch_y in test_generator:
        # Apply noise fusion
        fused_batch_x = fuse_with_noise_batch(batch_x)

        # Predict
        preds = model_uniform.predict(fused_batch_x, verbose=0)
        batch_preds = tf.argmax(preds, axis=1).numpy()

        y_true.append(batch_y)
        y_pred.append(batch_preds)

        # Exit when all samples processed
        if len(np.concatenate(y_true)) >= test_generator.samples:
            break

    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)

    # Calculate metrics
    acc = accuracy_score(y_true, y_pred)
    print(f"\nPost-Adversarial Accuracy (with Noise Fusion, ε={eps}): {acc:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))



Evaluating Adversarial Images for Epsilon = 0.007
Found 516 validated image filenames.

Post-Adversarial Accuracy (with Noise Fusion, ε=0.007): 0.8372

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.42      0.59        12
           1       0.71      1.00      0.83        12
           2       0.64      0.58      0.61        12
           3       0.67      0.67      0.67        12
           4       0.82      0.75      0.78        12
           5       0.48      1.00      0.65        12
           6       0.86      1.00      0.92        12
           7       1.00      0.67      0.80        12
           8       1.00      0.83      0.91        12
           9       1.00      0.75      0.86        12
          10       1.00      0.92      0.96        12
          11       0.80      1.00      0.89        12
          12       1.00      0.75      0.86        12
          13       1.00      1.00      1.00        12
          14 

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# Uniform distribuition on Pgd

In [13]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Load the CSV file
csv_path = '/content/Pgd_test/adversarial_images_PGD/adversarial_labels.csv'
df = pd.read_csv(csv_path)

# Define the noise fusion function
def fuse_with_noise_batch(batch, alpha=0.2):
    batch = tf.cast(batch, tf.float32)
    noise = tf.random.normal(tf.shape(batch), mean=0.0, stddev=255.0)
    fused = batch * (1.0 - alpha) + noise * alpha
    fused = tf.clip_by_value(fused, 0.0, 255.0)
    return fused  # Stay float32

# List of epsilon values to evaluate
epsilons = [0.007, 0.01, 0.03, 0.1]

for eps in epsilons:
    print(f"\n{'='*50}\nEvaluating Adversarial Images for Epsilon = {eps}\n{'='*50}")

    # Filter CSV for the current epsilon
    eps_df = df[df['epsilon'] == eps].copy()

    # Remove the 'eps_X.XXX/' prefix from filenames (if present)
    eps_df['filename'] = eps_df['filename'].str.replace(f'eps_{eps}/', '', regex=False)

    # Create ImageDataGenerator (no normalization)
    test_datagen = ImageDataGenerator()

    # Prepare test generator
    test_generator = test_datagen.flow_from_dataframe(
        dataframe=eps_df,
        directory=f'/content/Pgd_test/adversarial_images_PGD/eps_{eps}',  # Folder for current epsilon
        x_col='filename',
        y_col='label',
        target_size=(240, 240),
        batch_size=32,
        class_mode='raw',  # For integer labels
        shuffle=False
    )

    # Skip if no images found
    if test_generator.samples == 0:
        print(f" No images found for epsilon={eps}. Check paths or filenames.")
        continue

    # Predict batch-by-batch with noise fusion
    y_true = []
    y_pred = []

    for batch_x, batch_y in test_generator:
        # Apply noise fusion
        fused_batch_x = fuse_with_noise_batch(batch_x)

        # Predict
        preds = model_uniform.predict(fused_batch_x, verbose=0)
        batch_preds = tf.argmax(preds, axis=1).numpy()

        y_true.append(batch_y)
        y_pred.append(batch_preds)

        # Exit when all samples processed
        if len(np.concatenate(y_true)) >= test_generator.samples:
            break

    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)

    # Calculate metrics
    acc = accuracy_score(y_true, y_pred)
    print(f"\nPost-Adversarial Accuracy (with Noise Fusion, ε={eps}): {acc:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))



Evaluating Adversarial Images for Epsilon = 0.007
Found 516 validated image filenames.

Post-Adversarial Accuracy (with Noise Fusion, ε=0.007): 0.6899

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.33      0.47        12
           1       0.65      0.92      0.76        12
           2       0.58      0.58      0.58        12
           3       0.50      0.33      0.40        12
           4       0.60      0.50      0.55        12
           5       0.32      0.75      0.45        12
           6       0.50      0.33      0.40        12
           7       0.33      0.42      0.37        12
           8       0.56      0.42      0.48        12
           9       0.73      0.67      0.70        12
          10       1.00      0.58      0.74        12
          11       0.73      0.92      0.81        12
          12       1.00      0.58      0.74        12
          13       0.80      1.00      0.89        12
          14 

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


| Attack           | Distribution | ε (Epsilon) | α (Alpha) | Test Accuracy after Attack | Accuracy after Noise Fusion |
|:----------------:|:------------:|:-----------:|:---------:|:---------------------------:|:----------------------------:|
| FGSM             | Gaussian     | 0.007       | -         | 58%                         | 95.16%                      |
| FGSM             | Gaussian     | 0.01        | -         | 44%                         | 93.41%                      |
| FGSM             | Gaussian     | 0.03        | -         | 14%                         | 83.33%                      |
| FGSM             | Gaussian     | 0.1         | -         | 6%                          | 48.64%                      |
| PGD              | Gaussian     | 0.007       | 0.00175   | 35%                         | 93.22%                      |
| PGD              | Gaussian     | 0.01        | 0.0025    | 7%                          | 80.23%                      |
| PGD              | Gaussian     | 0.03        | 0.0075    | 3%                          | 67.05%                      |
| PGD              | Gaussian     | 0.1         | 0.025     | 0%                          | 42.83%                      |
| FGSM             | Uniform      | 0.007       | -         | 58%                         | 83.72%                      |
| FGSM             | Uniform      | 0.01        | -         | 44%                         | 81.59%                      |
| FGSM             | Uniform      | 0.03        | -         | 14%                         | 72.48%                      |
| FGSM             | Uniform      | 0.1         | -         | 6%                          | 42.64%                      |
| PGD              | Uniform      | 0.007       | 0.00175   | 35%                         | 68.99%                      |
| PGD              | Uniform      | 0.01        | 0.0025    | 7%                          | 39.92%                      |
| PGD              | Uniform      | 0.03        | 0.0075    | 3%                          | 25.39%                      |
| PGD              | Uniform      | 0.1         | 0.025     | 0%                          | 10.66%                      |
